In [ ]:
import pandas as pd

df = pd.read_csv("loan.csv")
df.shape

/tmp/ipykernel_17229/1865285561.py:3: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("loan.csv")


(2260668, 145)

In [ ]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

In [ ]:
df['loan_status'] = df['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})

In [ ]:
missing = df.isnull().mean() * 100
cols_to_drop = missing[missing > 80].index

df = df.drop(columns=cols_to_drop)

In [ ]:
features = [
    'loan_amnt',
    'term',
    'int_rate',
    'installment',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'dti',
    'revol_bal',
    'total_acc',
    'loan_status'
]

df = df[features]

In [ ]:
df['term'] = df['term'].str.extract('(\d+)').astype(int)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_17229/3024662541.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.extract('(\d+)').astype(int)


In [ ]:
df['emp_length'] = df['emp_length'].str.extract('(\d+)').astype(float)
df['emp_length'].fillna(0, inplace=True)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_17229/3563506733.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['emp_length'] = df['emp_length'].str.extract('(\d+)').astype(float)
/tmp/ipykernel_17229/3563506733.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['emp_length'].fillna(0, inplace=True)


In [ ]:
df['int_rate'] = df['int_rate'].astype(float)

In [ ]:
df['annual_inc'].fillna(df['annual_inc'].median(), inplace=True)
df['dti'].fillna(df['dti'].median(), inplace=True)

/tmp/ipykernel_17229/3342816645.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['annual_inc'].fillna(df['annual_inc'].median(), inplace=True)
/tmp/ipykernel_17229/3342816645.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('loan_status', axis=1)
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
probs = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

preds = model.predict(X_test)

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.87      0.63      0.73    208206
           1       0.30      0.64      0.41     52516

    accuracy                           0.63    260722
   macro avg       0.59      0.63      0.57    260722
weighted avg       0.76      0.63      0.67    260722

ROC-AUC: 0.6806947788619565


In [ ]:
X_test['risk'] = probs
X_test['expected_loss'] = X_test['loan_amnt'] * X_test['risk']

In [ ]:
def decision(p):
    if p < 0.2:
        return "Approve"
    elif p < 0.5:
        return "Review"
    else:
        return "Reject"

X_test['decision'] = X_test['risk'].apply(decision)


In [ ]:
import pandas as pd

# Interest rate impact
print(X_test.groupby(pd.qcut(X_test['int_rate'], 5))['risk'].mean())

# Income impact
print(X_test.groupby(pd.qcut(X_test['annual_inc'], 5))['risk'].mean())

# DTI impact
print(X_test.groupby(pd.qcut(X_test['dti'], 5))['risk'].mean())

/tmp/ipykernel_17229/2273648265.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(X_test.groupby(pd.qcut(X_test['int_rate'], 5))['risk'].mean())
/tmp/ipykernel_17229/2273648265.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(X_test.groupby(pd.qcut(X_test['annual_inc'], 5))['risk'].mean())


int_rate
(5.308999999999999, 8.99]    0.308539
(8.99, 11.53]                0.402916
(11.53, 13.99]               0.480888
(13.99, 16.99]               0.554974
(16.99, 30.99]               0.674346
Name: risk, dtype: float64
annual_inc
(-0.001, 42000.0]        0.546840
(42000.0, 57000.0]       0.505969
(57000.0, 74000.0]       0.485814
(74000.0, 100000.0]      0.461639
(100000.0, 9500000.0]    0.405031
Name: risk, dtype: float64
dti
(-0.001, 10.48]    0.434860
(10.48, 15.29]     0.454499
(15.29, 19.95]     0.476097
(19.95, 25.63]     0.500377
(25.63, 999.0]     0.544699
Name: risk, dtype: float64


/tmp/ipykernel_17229/2273648265.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(X_test.groupby(pd.qcut(X_test['dti'], 5))['risk'].mean())


In [ ]:
# Total risk if you approve everything
total_loss_all = X_test['expected_loss'].sum()

# Risk after applying your decision system
filtered = X_test[X_test['decision'] != 'Reject']
total_loss_filtered = filtered['expected_loss'].sum()

print("Total Loss (No System):", total_loss_all)
print("Total Loss (With System):", total_loss_filtered)

Total Loss (No System): 1853228170.3870127
Total Loss (With System): 773147155.5837562


In [ ]:
X_test.to_csv("final_output.csv", index=False)